# YC Bench W26: clean reproduction of the baseline result

Minimal, deterministic re-implementation of the computation in `YC_W26_Google_Mentions.ipynb`
(cells 2, 12, 13, 15, 20, 39 and 41). **Goal: fidelity, not improvement.**

```
during-batch mentions -> traction -> Velocity Score -> attention -> Pre-Demo Score (target)
                                                                  -> pre-application baseline -> evaluation
```

* No network access, scraping or APIs. Every input is a literal copied verbatim from the original notebook.
* No dataset on disk is modified. Section 8 only *reads* committed CSVs for cross-checks.
* The original notebook's exploratory material (Gini, Lorenz, rank plots, repeated charts, dead branches) is deliberately omitted.
* Variables are never reused: each stage has its own name.

In [1]:
import pandas as pd
from pathlib import Path

K = 20                  # size of the Top-K lists (original: top 10% of ~198 -> 20)
ATTENTION_WEIGHT = 0.05 # original cell 20: mentions_score = 0.05 * google_mentions
MOM_FACTOR = 10         # original cell 15: 1 + 10 * MoM
MOM_FLOOR = 0.1         # original cell 15: floor on the velocity multiplier

REPO_ROOT = Path.cwd().resolve().parent  # notebook lives in <repo>/notebooks/
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Google mentions during the batch (original cell 2)

198 domains with hard-coded Google result counts (collected after 2026-01-01; exact collection date is not recorded in the repository).
`google_mentions == 0` domains carry no attention signal. The original notebook drops them in cell 2 (`Loaded 184 active startup domains`)
and, independently, in cell 20 (`dropna` on the log of a zero Pre-Demo Score). Here the full 198 are kept for scoring and the active count is reported explicitly.

In [2]:
DURING_BATCH_MENTIONS_RAW = """
trycardinal.ai: 6
try.wideframe.com: 40
martini.film: 0
condor.energy: 0
resonate.audio: 0
shofo.ai: 103
wayco.ai: 131
unifold.io: 44
carrotlabs.ai: 28
aurorincad.com: 0
daivin.tech: 42
grazemate.com: 67
origami-robotics.com: 77
librarlabs.com: 41
therubric.ai: 9
zymbly.com: 8
ritivel.com: 25
usepolymorph.com: 10
pollinate.tech: 25
spotpay.ai: 564
voygr.tech: 34
mochacare.com: 10
tryvela.ai: 79
veriad.com: 9
talkingcomputers.ai: 9
copperlane.ai: 113
runcanary.ai: 31
usegrade.com: 47
silahq.com: 61
trymaven.com: 25
usemoda.ai: 5
thetokencompany.com: 115
beesafe.ai: 209
fed10.ai: 65
squid.energy: 0
usecrow.ai: 53
scoutout.ai: 43
bookoapp.com: 97
corelayer.com: 10
general.legal: 0
fort.cx: 222
getbalance.ai: 27
chasi.co: 57
tryremy.ai: 8
buildwithglue.com: 2
o11.ai: 1140
doomersion.com: 6
sequencemkts.com: 77
dittobio.com: 40
servo7.com: 7
arzule.com: 39
voxelenergy.com: 41
polymathlabs.ai: 38
bubblelab.ai: 483
heypocket.com: 755
byteport.io: 7
constellationspace.ai: 3
mendral.com: 55
axionorbital.space: 309
oximy.com: 64
caretta.so: 144
samora.ai: 171
humanarchive.ai: 156
forum.market: 0
sitefire.ai: 92
tryrobby.com: 6
ressl.ai: 500
emdash.sh: 43
congruent.io: 130
usereframe.ai: 10
valence.trade: 57
approxima.ai: 4
traverse.so: 0
revion.inc: 8
maywoodai.com: 30
fenrock.ai: 241
endclose.com: 5
usechamber.io: 99
patientdesk.ai: 899
visiblsemi.com: 235
syntheticsciences.ai: 117
seeing-systems.com: 10
jinba.io: 310
usecardboard.com: 165
pranadoc.com: 60
rhizomeai.com: 10
paxhistoria.co: 10500
hex.co: 0
trylaurence.com: 90
terminaluse.com: 76
overshoot.ai: 230
caj.al: 0
proximitty.ai: 39
stilta.com: 39
unisson.ai: 10
incidentfox.ai: 154
compresr.ai: 241
opalitehealth.com: 6
confluence.sh: 175
mangomedical.io: 3
paysponge.com: 58
terranox.ai: 158
ventura.ai: 0
orthogonal.com: 150
perfectly.so: 0
getfern.ai: 5
sparkles.dev: 221
eigenpal.com: 74
usesalus.ai: 10
arcline-ai.com: 40
ramain.ai: 97
milliray.com: 1
beyondreachlabs.com: 8
tryoctapulse.com: 10
codewisp.ai: 257
ashr.io: 130
aemon.ai: 95
luel.ai: 591
schedulingwiz.com: 3
robodock.tech: 25
skillsync.wiki: 77
mousecat.ai: 58
zatanna.ai: 154
autositu.com: 10
verdexai.com: 9
cumuluslabs.io: 63
tepali.com: 5
palus.finance: 278
21st.dev: 4260
avoice.co: 36
kytentech.com: 28
zerosettle.io: 76
usecarson.com: 39
openspec.dev: 239
pirislabs.io: 50
celltype.com: 41
trainburt.com: 25
envariant.ai: 6
runcascade.com: 10
beaconhealth.ai: 154
strandai.com: 32
tryclam.com: 29
protent.ai: 69
cofia.ai: 122
getnoetic.ai: 26
legalos.ai: 97
sentrial.com: 90
10xscience.com: 10
rev1.so: 84
shortkit.dev: 39
tryasimov.ai: 26
lexius.ai: 84
axisanalysis.com: 115
sonarly.com: 26
docurahealth.com: 8
lance.live: 0
runanywhere.ai: 971
lucenthq.com: 66
rumacare.com: 37
klausai.com: 62
usekita.com: 92
syntropy.io: 10
onerobot.io: 10
didit.me: 4350
pantainsure.com: 40
payna.com: 8
origin.bio: 0
hlaboratories.com: 10
helloeos.ai: 4
valgo.ai: 112
arcprize.org: 860
corvera.ai: 64
fullseam.com: 10
catchbackcards.com: 295
mantisbiotech.com: 7
haladir.com: 41
claimglide.com: 2
inviscidai.com: 25
ndea.com: 168
crosslayerlabs.com: 26
voltairlabs.com: 47
vectorlegal.com: 40
generalastro.com: 5
oxus-ai.com: 10
runcaptain.com: 30
agenticfabriq.com: 6
travoai.com: 4
velum-labs.com: 4
alt-x.co: 4
buttoncomputer.com: 41
pollen.cx: 9
autumnai.com: 6
overdrive.health: 179
returnsignals.com: 50
gru.space: 13300
usebidflow.com: 7
menza.ai: 115
tryq2q.com: 78
"""

mentions_during_df = pd.DataFrame(
    [line.split(": ") for line in DURING_BATCH_MENTIONS_RAW.strip().splitlines()],
    columns=["domain", "google_mentions_during"],
)
mentions_during_df["google_mentions_during"] = pd.to_numeric(mentions_during_df["google_mentions_during"])
assert mentions_during_df["domain"].is_unique

n_during_total = len(mentions_during_df)
n_during_active = int((mentions_during_df["google_mentions_during"] > 0).sum())
print(f"during-batch domains: total={n_during_total}  active (>0)={n_during_active}  zeros removed={n_during_total - n_during_active}")

during-batch domains: total=198  active (>0)=184  zeros removed=14


## 2. Traction signals (original cell 12-13)

Eleven startups with a hand-entered traction signal (source cited in the original notebook: a Lobster Capital article; not independently verified here).
`Base Traction Score = max(signal_value x signal_weight)` over the seven signal types.

In [3]:
TRACTION_SIGNALS = {'Startup': ['Pocket',
             'Avoice',
             'Mango Medical',
             'OctaPulse',
             'Luel',
             'Pax Historia',
             'Didit',
             '21st',
             'OpenSpec',
             'Samora AI',
             'Corvera'],
 'arr': [27000000, 240000, 0, 0, 0, 0, 0, 0, 0, 0, 300000],
 'pilot_revenue': [0, 0, 0, 100000, 0, 0, 0, 0, 0, 0, 0],
 'loi': [0, 0, 10000000, 0, 0, 0, 0, 0, 0, 0, 0],
 'signups': [0, 0, 0, 0, 190000, 0, 0, 1000000, 0, 0, 0],
 'active_users': [0, 0, 0, 0, 0, 35000, 0, 0, 0, 0, 0],
 'activity_volume': [0, 0, 0, 0, 0, 0, 4000000, 0, 0, 1000000, 0],
 'ecosystem_pull': [0, 0, 0, 0, 0, 0, 0, 0, 30000, 0, 0]}

SIGNAL_WEIGHTS = {'arr': 1.0,
 'pilot_revenue': 0.5,
 'loi': 0.2,
 'active_users': 0.4,
 'activity_volume': 0.25,
 'signups': 0.15,
 'ecosystem_pull': 0.1}

traction_df = pd.DataFrame(TRACTION_SIGNALS)
signal_cols = list(SIGNAL_WEIGHTS)
weighted_signals = traction_df[signal_cols].mul(pd.Series(SIGNAL_WEIGHTS))
traction_df["signal"] = weighted_signals.idxmax(axis=1)
traction_df["signal_value"] = [traction_df.loc[i, c] for i, c in traction_df["signal"].items()]
traction_df["base_score"] = weighted_signals.max(axis=1)

print(f"startups with traction: {len(traction_df)}")
print(traction_df[["Startup", "signal", "signal_value", "base_score"]]
      .sort_values("base_score", ascending=False).to_string(index=False))

startups with traction: 11
      Startup          signal  signal_value  base_score
       Pocket             arr      27000000  27000000.0
Mango Medical             loi      10000000   2000000.0
        Didit activity_volume       4000000   1000000.0
      Corvera             arr        300000    300000.0
    Samora AI activity_volume       1000000    250000.0
       Avoice             arr        240000    240000.0
         21st         signups       1000000    150000.0
    OctaPulse   pilot_revenue        100000     50000.0
         Luel         signups        190000     28500.0
 Pax Historia    active_users         35000     14000.0
     OpenSpec  ecosystem_pull         30000      3000.0


## 3. Velocity Score (original cell 15)

`Velocity_Score = Base_Score x max(1 + 10 x g_MoM, 0.1)`

The **0.1 floor belongs to the notebook implementation**; it is kept here because this notebook reproduces the notebook, not the paper's formula.
Only Pocket has a non-zero MoM (50%, from the Lobster Capital guide); all others default to 0.

In [5]:
MOM_GROWTH = {'Pocket': 0.5,
 'Avoice': 0.0,
 'OctaPulse': 0.0,
 'OpenSpec': 0.0,
 'Mango Medical': 0.0,
 'Pax Historia': 0.0,
 'Luel': 0.0,
 'Didit': 0.0,
 '21st': 0.0,
 'Samora AI': 0.0,
 'Corvera': 0.0}

traction_df["mom_growth"] = traction_df["Startup"].map(MOM_GROWTH).fillna(0)
traction_df["multiplier"] = (1 + MOM_FACTOR * traction_df["mom_growth"]).clip(lower=MOM_FLOOR)
traction_df["Velocity_Score"] = traction_df["base_score"] * traction_df["multiplier"]

print(f"multiplier range: min={traction_df['multiplier'].min()}  max={traction_df['multiplier'].max()}  (floor active: {bool((1 + MOM_FACTOR * traction_df['mom_growth'] < MOM_FLOOR).any())})")
print(traction_df[["Startup", "signal", "base_score", "mom_growth", "multiplier", "Velocity_Score"]]
      .sort_values("Velocity_Score", ascending=False).to_string(index=False))

multiplier range: min=1.0  max=6.0  (floor active: False)
      Startup          signal  base_score  mom_growth  multiplier  Velocity_Score
       Pocket             arr  27000000.0         0.5         6.0     162000000.0
Mango Medical             loi   2000000.0         0.0         1.0       2000000.0
        Didit activity_volume   1000000.0         0.0         1.0       1000000.0
      Corvera             arr    300000.0         0.0         1.0        300000.0
    Samora AI activity_volume    250000.0         0.0         1.0        250000.0
       Avoice             arr    240000.0         0.0         1.0        240000.0
         21st         signups    150000.0         0.0         1.0        150000.0
    OctaPulse   pilot_revenue     50000.0         0.0         1.0         50000.0
         Luel         signups     28500.0         0.0         1.0         28500.0
 Pax Historia    active_users     14000.0         0.0         1.0         14000.0
     OpenSpec  ecosystem_pull      3000.

## 4. Pre-Demo Day Score = target (original cell 20)

`attention_score = 0.05 x google_mentions_during` and `PreDemoScore = max(Velocity_Score, attention_score)`.
Startups without traction get `Velocity_Score = 0`. Domains whose score is 0 are discarded (original: `dropna` on `log10`).

In [6]:
STARTUP_TO_DOMAIN = {'Pocket': 'heypocket.com',
 'Avoice': 'avoice.co',
 'Mango Medical': 'mangomedical.io',
 'OctaPulse': 'tryoctapulse.com',
 'Luel': 'luel.ai',
 'Pax Historia': 'paxhistoria.co',
 'Didit': 'didit.me',
 '21st': '21st.dev',
 'OpenSpec': 'openspec.dev',
 'Samora AI': 'samora.ai',
 'Corvera': 'corvera.ai'}

traction_df["domain"] = traction_df["Startup"].map(STARTUP_TO_DOMAIN)
assert traction_df["domain"].notna().all() and traction_df["domain"].is_unique
assert set(traction_df["domain"]) <= set(mentions_during_df["domain"])

predemo_df = mentions_during_df.merge(traction_df[["domain", "Velocity_Score"]], on="domain", how="left")
predemo_df["Velocity_Score"] = predemo_df["Velocity_Score"].fillna(0)
predemo_df["attention_score"] = ATTENTION_WEIGHT * predemo_df["google_mentions_during"]
predemo_df["PreDemoScore"] = predemo_df[["Velocity_Score", "attention_score"]].max(axis=1)
predemo_df = predemo_df[predemo_df["PreDemoScore"] > 0]
predemo_df["dominant"] = (predemo_df["Velocity_Score"] >= predemo_df["attention_score"]).map({True: "traction", False: "attention"})
# deterministic order: score desc, domain asc as tie-break
predemo_df = predemo_df.sort_values(["PreDemoScore", "domain"], ascending=[False, True]).reset_index(drop=True)
predemo_df.insert(0, "rank", predemo_df.index + 1)

assert predemo_df.loc[K - 1, "PreDemoScore"] != predemo_df.loc[K, "PreDemoScore"], "tie at the Top-K boundary"
print(f"domains with a Pre-Demo Score > 0: {len(predemo_df)}  (matches the original's 'Total startups with signal')")

target_top20 = predemo_df.head(K).copy()
print(target_top20[["rank", "domain", "google_mentions_during", "Velocity_Score", "attention_score", "PreDemoScore", "dominant"]]
      .to_string(index=False))

domains with a Pre-Demo Score > 0: 184  (matches the original's 'Total startups with signal')
 rank           domain  google_mentions_during  Velocity_Score  attention_score  PreDemoScore  dominant
    1    heypocket.com                     755     162000000.0            37.75  162000000.00  traction
    2  mangomedical.io                       3       2000000.0             0.15    2000000.00  traction
    3         didit.me                    4350       1000000.0           217.50    1000000.00  traction
    4       corvera.ai                      64        300000.0             3.20     300000.00  traction
    5        samora.ai                     171        250000.0             8.55     250000.00  traction
    6        avoice.co                      36        240000.0             1.80     240000.00  traction
    7         21st.dev                    4260        150000.0           213.00     150000.00  traction
    8 tryoctapulse.com                      10         50000.0            

## 5. Pre-application baseline (original cells 38-39)

Baseline = Google mentions for the same 198 domains in the window **2025-08-17 to 2025-10-31** (67 days), before the YC W26 application deadline.

**Raw vs. hard-coded data.** The values below are the ones hard-coded in the original notebook (cell 39), which is what produced the published 14/20.
They differ from the repository file `yc_mentions_early.csv` (direct SerpAPI output): see section 8 for the measured differences.
The rule that turned raw counts into these values is **not documented anywhere in the repository**. It is not applied or reconstructed here.

In [7]:
PRE_APPLICATION_MENTIONS_RAW = """
trycardinal.ai: 43
try.wideframe.com: 0
martini.film: 0
condor.energy: 0
resonate.audio: 0
shofo.ai: 3
wayco.ai: 60
unifold.io: 7
carrotlabs.ai: 27
aurorincad.com: 0
daivin.tech: 0
grazemate.com: 10
origami-robotics.com: 5
librarlabs.com: 1
therubric.ai: 0
zymbly.com: 0
ritivel.com: 2
usepolymorph.com: 2
pollinate.tech: 1
spotpay.ai: 271
voygr.tech: 0
mochacare.com: 0
tryvela.ai: 3
veriad.com: 0
talkingcomputers.ai: 0
copperlane.ai: 9
runcanary.ai: 1
usegrade.com: 0
silahq.com: 2
trymaven.com: 1
usemoda.ai: 0
thetokencompany.com: 0
beesafe.ai: 29
fed10.ai: 6
squid.energy: 0
usecrow.ai: 8
scoutout.ai: 47
bookoapp.com: 148
corelayer.com: 1
general.legal: 0
fort.cx: 10
getbalance.ai: 0
chasi.co: 10
tryremy.ai: 0
buildwithglue.com: 9
o11.ai: 450
doomersion.com: 3
sequencemkts.com: 0
dittobio.com: 66
servo7.com: 1
arzule.com: 0
voxelenergy.com: 0
polymathlabs.ai: 2
bubblelab.ai: 194
heypocket.com: 281
byteport.io: 0
constellationspace.ai: 1
mendral.com: 3
axionorbital.space: 189
oximy.com: 0
caretta.so: 0
samora.ai: 74
humanarchive.ai: 5
forum.market: 0
sitefire.ai: 6
tryrobby.com: 0
ressl.ai: 276
emdash.sh: 6
congruent.io: 0
usereframe.ai: 0
valence.trade: 8
approxima.ai: 0
traverse.so: 0
revion.inc: 1
maywoodai.com: 1
fenrock.ai: 0
endclose.com: 0
usechamber.io: 1
patientdesk.ai: 279
visiblsemi.com: 0
syntheticsciences.ai: 81
seeing-systems.com: 1
jinba.io: 46
usecardboard.com: 4
pranadoc.com: 4
rhizomeai.com: 5
paxhistoria.co: 4450
hex.co: 0
trylaurence.com: 1
terminaluse.com: 0
overshoot.ai: 47
caj.al: 0
proximitty.ai: 1
stilta.com: 2
unisson.ai: 2
incidentfox.ai: 1
compresr.ai: 6
opalitehealth.com: 0
confluence.sh: 72
mangomedical.io: 2
paysponge.com: 0
terranox.ai: 60
ventura.ai: 0
orthogonal.com: 34
perfectly.so: 0
getfern.ai: 0
sparkles.dev: 4
eigenpal.com: 1
usesalus.ai: 6
arcline-ai.com: 0
ramain.ai: 3
milliray.com: 0
beyondreachlabs.com: 1
tryoctapulse.com: 2
codewisp.ai: 4
ashr.io: 1
aemon.ai: 2
luel.ai: 124
schedulingwiz.com: 5
robodock.tech: 2
skillsync.wiki: 5
mousecat.ai: 1
zatanna.ai: 0
autositu.com: 9
verdexai.com: 2
cumuluslabs.io: 1
tepali.com: 0
palus.finance: 10
21st.dev: 1880
avoice.co: 0
kytentech.com: 1
zerosettle.io: 0
usecarson.com: 0
openspec.dev: 87
pirislabs.io: 3
celltype.com: 0
trainburt.com: 1
envariant.ai: 0
runcascade.com: 0
beaconhealth.ai: 0
strandai.com: 0
tryclam.com: 1
protent.ai: 3
cofia.ai: 1
getnoetic.ai: 0
legalos.ai: 62
sentrial.com: 1
10xscience.com: 218
rev1.so: 7
shortkit.dev: 0
tryasimov.ai: 3
lexius.ai: 3
axisanalysis.com: 0
sonarly.com: 1
docurahealth.com: 34
lance.live: 0
runanywhere.ai: 182
lucenthq.com: 4
rumacare.com: 2
klausai.com: 2
usekita.com: 2
syntropy.io: 8
onerobot.io: 0
didit.me: 2940
pantainsure.com: 3
payna.com: 1
origin.bio: 0
hlaboratories.com: 1
helloeos.ai: 1
valgo.ai: 42
arcprize.org: 784
corvera.ai: 5
fullseam.com: 0
catchbackcards.com: 150
mantisbiotech.com: 1
haladir.com: 2
claimglide.com: 1
inviscidai.com: 44
ndea.com: 51
crosslayerlabs.com: 0
voltairlabs.com: 4
vectorlegal.com: 0
generalastro.com: 0
oxus-ai.com: 0
runcaptain.com: 6
agenticfabriq.com: 2
travoai.com: 3
velum-labs.com: 3
alt-x.co: 3
buttoncomputer.com: 2
pollen.cx: 2
autumnai.com: 1
overdrive.health: 0
returnsignals.com: 0
gru.space: 4340
usebidflow.com: 0
menza.ai: 259
tryq2q.com: 2
"""

mentions_pre_application_df = pd.DataFrame(
    [line.split(": ") for line in PRE_APPLICATION_MENTIONS_RAW.strip().splitlines()],
    columns=["domain", "google_mentions_pre_application"],
)
mentions_pre_application_df["google_mentions_pre_application"] = pd.to_numeric(
    mentions_pre_application_df["google_mentions_pre_application"]
)
assert mentions_pre_application_df["domain"].is_unique
assert set(mentions_pre_application_df["domain"]) == set(mentions_during_df["domain"]), "same 198 domains expected"

n_pre_total = len(mentions_pre_application_df)
n_pre_active = int((mentions_pre_application_df["google_mentions_pre_application"] > 0).sum())
print(f"pre-application domains: total={n_pre_total}  active (>0)={n_pre_active}  zeros={n_pre_total - n_pre_active}")
print("(same 198-domain batch as section 1; 130 vs 184 are different measurements of the same batch, not different batch sizes)")

baseline_ranked = (
    mentions_pre_application_df[mentions_pre_application_df["google_mentions_pre_application"] > 0]
    .sort_values(["google_mentions_pre_application", "domain"], ascending=[False, True])
    .reset_index(drop=True)
)
baseline_ranked.insert(0, "rank", baseline_ranked.index + 1)
assert (baseline_ranked.loc[K - 1, "google_mentions_pre_application"]
        != baseline_ranked.loc[K, "google_mentions_pre_application"]), "tie at the Top-K boundary"

baseline_top20 = baseline_ranked.head(K).copy()
print(baseline_top20.to_string(index=False))

pre-application domains: total=198  active (>0)=130  zeros=68
(same 198-domain batch as section 1; 130 vs 184 are different measurements of the same batch, not different batch sizes)
 rank               domain  google_mentions_pre_application
    1       paxhistoria.co                             4450
    2            gru.space                             4340
    3             didit.me                             2940
    4             21st.dev                             1880
    5         arcprize.org                              784
    6               o11.ai                              450
    7        heypocket.com                              281
    8       patientdesk.ai                              279
    9             ressl.ai                              276
   10           spotpay.ai                              271
   11             menza.ai                              259
   12       10xscience.com                              218
   13         bubblelab.ai           

## 6. Evaluation (original cell 41)

`Precision@20 = |baseline_top20 ∩ target_top20| / 20`.

In [8]:
target_set = set(target_top20["domain"])
baseline_set = set(baseline_top20["domain"])
captured = baseline_set & target_set

captured_df = (target_top20[["rank", "domain", "dominant"]]
               .rename(columns={"rank": "target_rank"})
               .merge(baseline_top20[["rank", "domain"]].rename(columns={"rank": "baseline_rank"}), on="domain"))
print(captured_df.to_string(index=False))

precision_at_20 = len(captured) / K
print(f"\nPrecision@{K} = {len(captured)} / {K} = {precision_at_20:.0%}")
print("missed by baseline:", sorted(target_set - baseline_set))

 target_rank         domain  dominant  baseline_rank
           1  heypocket.com  traction              7
           3       didit.me  traction              3
           7       21st.dev  traction              4
           9        luel.ai  traction             18
          10 paxhistoria.co  traction              1
          11   openspec.dev  traction             19
          12      gru.space attention              2
          13         o11.ai attention              6
          14 runanywhere.ai attention             15
          15 patientdesk.ai attention              8
          16   arcprize.org attention              5
          17     spotpay.ai attention             10
          18       ressl.ai attention              9
          19   bubblelab.ai attention             13

Precision@20 = 14 / 20 = 70%
missed by baseline: ['avoice.co', 'corvera.ai', 'jinba.io', 'mangomedical.io', 'samora.ai', 'tryoctapulse.com']


### Recall@11 and lift

* **Recall@11, candidate definition (not confirmed).** No code in the repository computes it; the README only states "Recall@11 = 55%".
  The only reading supported by repository artifacts is: *of the 11 startups with traction, how many are in the baseline Top-20*. It is computed below and labelled as a candidate.
  It is **not** claimed to be the paper's definition.
* **Lift: not reproducible.** No definition is available in the repository. The published values are listed separately and are not derived here.

In [9]:
traction_domains = set(traction_df["domain"])
recall_captured = sorted(traction_domains & baseline_set)
recall_candidate = len(recall_captured) / len(traction_domains)
print(f"Recall@11 (CANDIDATE definition) = {len(recall_captured)} / {len(traction_domains)} = {recall_candidate:.0%}")
print("traction startups in baseline Top-20:", recall_captured)
print("traction startups missed:           ", sorted(traction_domains - baseline_set))

published = pd.DataFrame(
    {"Precision@20": ["70%", "30% (6/20)"], "Recall@11": ["55%", "55% (6/11)"], "Lift": ["7x", "2.75x"]},
    index=["README (since 2026-04-02)", "README 2026-03-30..04-02; arXiv 2604.02378 per project owner (paper not consulted)"],
)
reproduced = pd.DataFrame(
    {"Precision@20": [f"{precision_at_20:.0%} ({len(captured)}/{K})"],
     "Recall@11": [f"{recall_candidate:.0%} ({len(recall_captured)}/{len(traction_domains)}) candidate, unconfirmed"],
     "Lift": ["not reproducible (no definition)"]},
    index=["this notebook"],
)
print("\n" + pd.concat([published, reproduced]).to_string())

Recall@11 (CANDIDATE definition) = 6 / 11 = 55%
traction startups in baseline Top-20: ['21st.dev', 'didit.me', 'heypocket.com', 'luel.ai', 'openspec.dev', 'paxhistoria.co']
traction startups missed:            ['avoice.co', 'corvera.ai', 'mangomedical.io', 'samora.ai', 'tryoctapulse.com']

                                                                                   Precision@20                          Recall@11                              Lift
README (since 2026-04-02)                                                                   70%                                55%                                7x
README 2026-03-30..04-02; arXiv 2604.02378 per project owner (paper not consulted)   30% (6/20)                         55% (6/11)                             2.75x
this notebook                                                                       70% (14/20)  55% (6/11) candidate, unconfirmed  not reproducible (no definition)


## 7. Invariants against the original notebook

Values below are the outputs stored in the original notebook (cells 2, 20, 39 and 41).

In [10]:
assert n_during_total == 198 and n_during_active == 184
assert len(predemo_df) == 184
assert len(traction_df) == 11
assert n_pre_total == 198 and n_pre_active == 130
assert (len(captured), K) == (14, 20)
print("all invariants match the original notebook outputs: 184 / 184 / 11 / 130 / 14 of 20")

all invariants match the original notebook outputs: 184 / 184 / 11 / 130 / 14 of 20


## 8. Cross-checks against committed artifacts (read-only)

These do not feed the result. They document how this reproduction relates to files in the repository.

In [11]:
def read_csv_or_none(name):
    path = REPO_ROOT / name
    return pd.read_csv(path) if path.exists() else None

predemo_csv = read_csv_or_none("yc_w26_pre_demo_scores.csv")
if predemo_csv is not None:
    merged = predemo_csv.merge(predemo_df[["domain", "google_mentions_during", "Velocity_Score", "PreDemoScore"]], on="domain", how="outer", indicator=True)
    same = ((merged["_merge"] == "both")
            & (merged["google_mentions"] == merged["google_mentions_during"])
            & ((merged["hybrid_score"] - merged["PreDemoScore"]).abs() < 1e-9)).sum()
    print(f"yc_w26_pre_demo_scores.csv: {len(predemo_csv)} rows; identical (mentions and score) to section 4 on {same} rows; unmatched rows: {(merged['_merge'] != 'both').sum()}")

traction_csv = read_csv_or_none("yc_w26_traction.csv")
if traction_csv is not None:
    t = traction_csv.merge(traction_df[["Startup", "Velocity_Score"]], on="Startup", suffixes=("_csv", "_here"))
    print(f"yc_w26_traction.csv: Velocity_Score identical on {int(((t['Velocity_Score_csv'] - t['Velocity_Score_here']).abs() < 1e-9).sum())} / {len(traction_csv)} rows")

early_csv = read_csv_or_none("yc_mentions_early.csv")
if early_csv is not None:
    cmp_df = mentions_pre_application_df.merge(early_csv.rename(columns={"google_mentions": "raw_csv"}), on="domain")
    diff = cmp_df[cmp_df["google_mentions_pre_application"] != cmp_df["raw_csv"]]
    csv_top20 = set(early_csv.sort_values("google_mentions", ascending=False).head(K)["domain"])
    print(f"yc_mentions_early.csv (raw SerpAPI output): {len(diff)} of {len(cmp_df)} rows differ from the hard-coded baseline used here;"
          f" {int((diff['google_mentions_pre_application'] == 0).sum())} of them are 0 here and positive in the CSV.")
    print(f"Top-20 of the raw CSV vs target_top20: {len(csv_top20 & target_set)} / {K} (not used for the result)")

yc_w26_pre_demo_scores.csv: 184 rows; identical (mentions and score) to section 4 on 184 rows; unmatched rows: 0
yc_w26_traction.csv: Velocity_Score identical on 11 / 11 rows
yc_mentions_early.csv (raw SerpAPI output): 58 of 198 rows differ from the hard-coded baseline used here; 56 of them are 0 here and positive in the CSV.
Top-20 of the raw CSV vs target_top20: 1 / 20 (not used for the result)
